# Aplicaciones de Multi-Head Attention

## 1. Introducción

Multi-Head Attention es uno de los componentes fundamentales de la
arquitectura Transformer.

Su función es permitir que cada elemento de una secuencia pueda
considerar información de otros elementos, aprendiendo diferentes
tipos de relaciones mediante múltiples cabezas de atención.

Aunque inicialmente fue propuesta para procesamiento de lenguaje
natural, actualmente puede utilizarse en diferentes tipos de datos.

En este cuadernillo se presentan cinco posibles aplicaciones:

1. Clasificación de texto.
2. Traducción automática.
3. Clasificación de imágenes.
4. Reconocimiento de voz.
5. Predicción de series temporales.

Para cada aplicación se muestra un ejemplo de código en PyTorch y se
explica cómo interviene Multi-Head Attention.

## 2. Librerías y configuración

Se utilizará PyTorch para implementar los ejemplos de
Multi-Head Attention.

Los ejemplos se enfocan en mostrar el funcionamiento del mecanismo,
por lo que se utilizarán datos sintéticos cuando no sea necesario
utilizar un dataset completo.

In [1]:
import torch
import torch.nn as nn

print("PyTorch:", torch.__version__)

PyTorch: 2.13.0+cpu


# 3. Ejemplo 1 — Clasificación de texto

La Multi-Head Attention puede utilizarse para analizar las relaciones
entre diferentes palabras de una secuencia.

En clasificación de texto, una oración puede convertirse en una
secuencia de tokens. Cada token se representa mediante un vector y
la atención permite incorporar información contextual de los demás
tokens.

Por ejemplo, una oración puede clasificarse como positiva o negativa.

### Representación de los tokens

Para simplificar el ejemplo, cada palabra será representada mediante
un vector de características.

La entrada tendrá la forma:

[batch_size, sequence_length, embedding_dim]

donde:

- batch_size: cantidad de oraciones.
- sequence_length: cantidad de tokens por oración.
- embedding_dim: dimensión del vector de cada token.

In [2]:
batch_size = 2
sequence_length = 8
embedding_dim = 64

x = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim
)

print("Forma de entrada:", x.shape)

Forma de entrada: torch.Size([2, 8, 64])


### Multi-Head Self-Attention

Se utilizará Multi-Head Attention con 8 cabezas.

Como se trata de Self-Attention, la misma secuencia se utiliza para
generar Query, Key y Value:

Q = X
K = X
V = X

Cada cabeza aprende diferentes relaciones entre los tokens.

In [3]:
num_heads = 8

attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)

output, weights = attention(
    x,
    x,
    x
)

print("Entrada:", x.shape)
print("Salida:", output.shape)
print("Pesos de atención:", weights.shape)

Entrada: torch.Size([2, 8, 64])
Salida: torch.Size([2, 8, 64])
Pesos de atención: torch.Size([2, 8, 8])


### Clasificación

Después de aplicar Multi-Head Attention, podemos utilizar una de las
representaciones obtenidas para realizar la clasificación.

En este ejemplo se utiliza el primer token como representación
resumida de la oración.

In [4]:
# Tomamos el primer token
sequence_representation = output[:, 0, :]

print(
    "Representación para clasificación:",
    sequence_representation.shape
)

Representación para clasificación: torch.Size([2, 64])


In [5]:
classifier = nn.Linear(
    embedding_dim,
    2
)

logits = classifier(
    sequence_representation
)

print("Salida del clasificador:", logits.shape)

Salida del clasificador: torch.Size([2, 2])


### Código completo

El siguiente código reúne los pasos anteriores en un ejemplo sencillo
de clasificación de texto utilizando Multi-Head Attention.

In [6]:
import torch
import torch.nn as nn


# Parámetros
batch_size = 2
sequence_length = 8
embedding_dim = 64
num_heads = 8
num_classes = 2


# Datos de entrada
x = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim
)


# Multi-Head Attention
attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)


# Self-Attention
output, weights = attention(
    x,
    x,
    x
)


# Representación de la secuencia
sequence_representation = output[:, 0, :]


# Clasificador
classifier = nn.Linear(
    embedding_dim,
    num_classes
)


# Predicción
logits = classifier(
    sequence_representation
)


print("Entrada:", x.shape)
print("Salida de Attention:", output.shape)
print("Representación:", sequence_representation.shape)
print("Predicción:", logits.shape)

Entrada: torch.Size([2, 8, 64])
Salida de Attention: torch.Size([2, 8, 64])
Representación: torch.Size([2, 64])
Predicción: torch.Size([2, 2])


### ¿Cómo se utiliza Multi-Head Attention?

En este ejemplo, Multi-Head Attention recibe una secuencia de tokens
y permite que cada token considere información de los demás tokens.

Se utiliza Self-Attention porque Q, K y V provienen de la misma
secuencia.

Las diferentes cabezas pueden aprender distintos tipos de relaciones
entre las palabras. Después, sus resultados se combinan para producir
una representación contextual de la oración.

Finalmente, esta representación se utiliza como entrada de un
clasificador para determinar la categoría del texto.

# 4. Ejemplo 2 — Traducción automática

Multi-Head Attention puede utilizarse en sistemas de traducción
automática para relacionar las palabras de una oración de entrada
con las palabras que se están generando en la traducción.

En un Transformer encoder-decoder, el decoder utiliza atención para
consultar la información producida por el encoder.

Por ejemplo:

"El gato está sobre la mesa"

            ↓

      Transformer

            ↓

"The cat is on the table"

### Representación de la secuencia

El encoder recibe la oración original y genera una representación
vectorial para cada token.

El decoder contiene los tokens que se han generado hasta el momento.

En este ejemplo utilizaremos:

- 6 tokens en la entrada.
- 4 tokens generados.
- Embeddings de 64 dimensiones.
- 8 cabezas de atención.

In [7]:
import torch
import torch.nn as nn

batch_size = 1
encoder_sequence = 6
decoder_sequence = 4
embedding_dim = 64
num_heads = 8

# Representación generada por el encoder
encoder_output = torch.randn(
    batch_size,
    encoder_sequence,
    embedding_dim
)

# Entrada actual del decoder
decoder_input = torch.randn(
    batch_size,
    decoder_sequence,
    embedding_dim
)

print("Encoder:", encoder_output.shape)
print("Decoder:", decoder_input.shape)

Encoder: torch.Size([1, 6, 64])
Decoder: torch.Size([1, 4, 64])


### Cross-Attention

En Cross-Attention, Query proviene del decoder, mientras que Key y
Value provienen de la salida del encoder.

Por lo tanto:

Q = Decoder

K = Encoder

V = Encoder

Esto permite que el decoder consulte diferentes partes de la oración
de entrada mientras genera la traducción.

In [8]:
attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)

output, weights = attention(
    decoder_input,   # Query
    encoder_output,  # Key
    encoder_output   # Value
)

print("Salida:", output.shape)
print("Pesos de atención:", weights.shape)

Salida: torch.Size([1, 4, 64])
Pesos de atención: torch.Size([1, 4, 6])


### Interpretación

Los pesos de atención indican qué partes de la representación del
encoder son más relevantes para cada posición del decoder.

De esta forma, el modelo puede aprender relaciones entre la oración
original y la traducción que está generando.

In [9]:
vocab_size = 10000

output_layer = nn.Linear(
    embedding_dim,
    vocab_size
)

logits = output_layer(output)

print("Salida del modelo:", logits.shape)

Salida del modelo: torch.Size([1, 4, 10000])


### Código completo

El siguiente ejemplo muestra de forma simplificada cómo se puede
utilizar Multi-Head Attention para implementar la parte de
Cross-Attention de un modelo de traducción.

In [10]:
import torch
import torch.nn as nn


# Parámetros
batch_size = 1
encoder_sequence = 6
decoder_sequence = 4
embedding_dim = 64
num_heads = 8
vocab_size = 10000


# Representación del encoder
encoder_output = torch.randn(
    batch_size,
    encoder_sequence,
    embedding_dim
)


# Entrada del decoder
decoder_input = torch.randn(
    batch_size,
    decoder_sequence,
    embedding_dim
)


# Multi-Head Attention
attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)


# Cross-Attention
output, weights = attention(
    decoder_input,   # Query
    encoder_output,  # Key
    encoder_output   # Value
)


# Capa de salida
output_layer = nn.Linear(
    embedding_dim,
    vocab_size
)


logits = output_layer(output)


print("Encoder:", encoder_output.shape)
print("Decoder:", decoder_input.shape)
print("Attention:", output.shape)
print("Logits:", logits.shape)

Encoder: torch.Size([1, 6, 64])
Decoder: torch.Size([1, 4, 64])
Attention: torch.Size([1, 4, 64])
Logits: torch.Size([1, 4, 10000])


### ¿Cómo se utiliza Multi-Head Attention?

En este ejemplo se utiliza Cross-Attention para conectar el decoder
con la información producida por el encoder.

El decoder genera los Query, mientras que el encoder proporciona
los Key y Value.

De esta manera, cada posición del decoder puede seleccionar la
información más relevante de la oración original.

Después de la atención, una capa lineal transforma la representación
obtenida en puntuaciones para cada palabra del vocabulario.

# 5. Ejemplo 3 — Clasificación de imágenes

Multi-Head Attention también puede utilizarse para trabajar con
imágenes.

En un Vision Transformer, la imagen se divide en pequeños patches.
Cada patch se convierte en un vector y se trata como un token.

De esta forma, Multi-Head Attention puede aprender relaciones entre
las diferentes regiones de la imagen.

### Representación de los patches

Una imagen de 224 × 224 píxeles con 3 canales se divide en
196 patches de 16 × 16 píxeles.

Cada patch contiene:

16 × 16 × 3 = 768 valores.

Estos valores se proyectan mediante una capa lineal para obtener
un embedding de 64 dimensiones.

In [11]:
import torch
import torch.nn as nn

batch_size = 2
num_patches = 196
patch_size = 768
embedding_dim = 64

# Representación de los patches
patches = torch.randn(
    batch_size,
    num_patches,
    patch_size
)

print("Patches:", patches.shape)

Patches: torch.Size([2, 196, 768])


In [12]:
patch_embedding = nn.Linear(
    patch_size,
    embedding_dim
)

x = patch_embedding(patches)

print("Embeddings:", x.shape)

Embeddings: torch.Size([2, 196, 64])


### Multi-Head Self-Attention

Cada patch se considera un token.

La Self-Attention permite que cada patch pueda relacionarse con
los demás patches de la imagen.

Como Q, K y V proceden de la misma secuencia:

Q = X
K = X
V = X

In [13]:
num_heads = 8

attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)

output, weights = attention(
    x,
    x,
    x
)

print("Entrada:", x.shape)
print("Salida:", output.shape)
print("Pesos:", weights.shape)

Entrada: torch.Size([2, 196, 64])
Salida: torch.Size([2, 196, 64])
Pesos: torch.Size([2, 196, 196])


### Classification Token

Se puede añadir un token especial al inicio de la secuencia.

Este token recibe información de los demás patches mediante
Self-Attention y posteriormente puede utilizarse como representación
de toda la imagen.

In [14]:
cls_token = torch.randn(
    batch_size,
    1,
    embedding_dim
)

x = torch.cat(
    [cls_token, x],
    dim=1
)

print("Secuencia con CLS:", x.shape)

Secuencia con CLS: torch.Size([2, 197, 64])


In [15]:
output, weights = attention(
    x,
    x,
    x
)

print(output.shape)

torch.Size([2, 197, 64])


In [16]:
cls_output = output[:, 0, :]

print("CLS:", cls_output.shape)

CLS: torch.Size([2, 64])


In [17]:
num_classes = 15

classifier = nn.Linear(
    embedding_dim,
    num_classes
)

logits = classifier(cls_output)

print("Predicción:", logits.shape)

Predicción: torch.Size([2, 15])


### Código completo

El siguiente ejemplo muestra una implementación simplificada del uso
de Multi-Head Attention para clasificación de imágenes mediante la
idea de Vision Transformer.

In [18]:
import torch
import torch.nn as nn


# Parámetros
batch_size = 2
num_patches = 196
patch_size = 768
embedding_dim = 64
num_heads = 8
num_classes = 15


# Patches de la imagen
patches = torch.randn(
    batch_size,
    num_patches,
    patch_size
)


# Patch Embedding
patch_embedding = nn.Linear(
    patch_size,
    embedding_dim
)

x = patch_embedding(patches)


# CLS token
cls_token = torch.randn(
    batch_size,
    1,
    embedding_dim
)

x = torch.cat(
    [cls_token, x],
    dim=1
)


# Multi-Head Self-Attention
attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)

output, weights = attention(
    x,
    x,
    x
)


# CLS token
cls_output = output[:, 0, :]


# Clasificador
classifier = nn.Linear(
    embedding_dim,
    num_classes
)

logits = classifier(
    cls_output
)


print("Patches:", patches.shape)
print("Embeddings + CLS:", x.shape)
print("Attention:", output.shape)
print("CLS:", cls_output.shape)
print("Predicción:", logits.shape)

Patches: torch.Size([2, 196, 768])
Embeddings + CLS: torch.Size([2, 197, 64])
Attention: torch.Size([2, 197, 64])
CLS: torch.Size([2, 64])
Predicción: torch.Size([2, 15])


### ¿Cómo se utiliza Multi-Head Attention?

En este ejemplo, cada patch de la imagen se considera un token.

Multi-Head Self-Attention permite que cada patch pueda relacionarse
con los demás patches y obtener información del contexto global de
la imagen.

El CLS token recopila información de la secuencia y su representación
se utiliza posteriormente para realizar la clasificación.

# 6. Ejemplo 4 — Reconocimiento de voz

Multi-Head Attention también puede utilizarse para procesar señales
de audio y reconocimiento automático del habla.

El audio puede dividirse en pequeños segmentos temporales. Cada
segmento se representa mediante un vector de características y se
trata como un token.

Multi-Head Attention permite relacionar diferentes momentos de la
señal para obtener información contextual.

### Representación del audio

Para simplificar el ejemplo, se supone que el audio ya fue convertido
en una secuencia de características.

Cada instante temporal se representa mediante un vector de 80 valores.

La entrada tendrá la forma:

[batch_size, sequence_length, feature_dim]

In [19]:
import torch
import torch.nn as nn

batch_size = 4
sequence_length = 100
feature_dim = 80
num_heads = 8

# Características extraídas del audio
audio = torch.randn(
    batch_size,
    sequence_length,
    feature_dim
)

print("Entrada:", audio.shape)

Entrada: torch.Size([4, 100, 80])


### Multi-Head Self-Attention

En este ejemplo utilizamos Self-Attention porque los Query, Key y
Value proceden de la misma secuencia de características del audio.

Por lo tanto:

Q = X
K = X
V = X

Cada segmento temporal puede relacionarse con los demás segmentos
para incorporar información del contexto.

In [20]:
attention = nn.MultiheadAttention(
    embed_dim=feature_dim,
    num_heads=num_heads,
    batch_first=True
)

output, weights = attention(
    audio,
    audio,
    audio
)

print("Entrada:", audio.shape)
print("Salida:", output.shape)

Entrada: torch.Size([4, 100, 80])
Salida: torch.Size([4, 100, 80])


### Relaciones temporales

Un sonido puede depender del contexto de otros sonidos que aparecen
antes o después.

Multi-Head Attention permite que cada segmento pueda consultar otros
segmentos de la secuencia y determinar cuáles son relevantes.

Por ejemplo, una parte del audio puede relacionarse con sonidos que
aparecieron varios instantes antes.

### Representación del audio

Después de aplicar Multi-Head Attention podemos combinar las
representaciones de los diferentes segmentos para obtener una
representación global del audio.

En este ejemplo utilizaremos el promedio de todos los segmentos.

In [21]:
# Promedio de todos los segmentos temporales
audio_representation = output.mean(dim=1)

print(
    "Representación:",
    audio_representation.shape
)

Representación: torch.Size([4, 80])


In [22]:
num_classes = 10

classifier = nn.Linear(
    feature_dim,
    num_classes
)

logits = classifier(
    audio_representation
)

print("Predicción:", logits.shape)

Predicción: torch.Size([4, 10])


### Código completo

El siguiente ejemplo muestra una implementación simplificada de
Multi-Head Attention aplicada a una secuencia de características
extraídas de audio.

In [23]:
import torch
import torch.nn as nn


# Parámetros
batch_size = 4
sequence_length = 100
feature_dim = 80
num_heads = 8
num_classes = 10


# Características del audio
audio = torch.randn(
    batch_size,
    sequence_length,
    feature_dim
)


# Multi-Head Attention
attention = nn.MultiheadAttention(
    embed_dim=feature_dim,
    num_heads=num_heads,
    batch_first=True
)


# Self-Attention
output, weights = attention(
    audio,
    audio,
    audio
)


# Representación global
audio_representation = output.mean(
    dim=1
)


# Clasificador
classifier = nn.Linear(
    feature_dim,
    num_classes
)


# Predicción
logits = classifier(
    audio_representation
)


print("Entrada:", audio.shape)
print("Attention:", output.shape)
print("Representación:", audio_representation.shape)
print("Predicción:", logits.shape)

Entrada: torch.Size([4, 100, 80])
Attention: torch.Size([4, 100, 80])
Representación: torch.Size([4, 80])
Predicción: torch.Size([4, 10])


### ¿Cómo se utiliza Multi-Head Attention?

En este ejemplo, cada segmento temporal del audio se considera un
token.

Multi-Head Self-Attention permite relacionar los diferentes
segmentos de la señal y aprender dependencias temporales.

La salida de Attention contiene una representación contextual para
cada segmento. Estas representaciones se combinan para obtener una
representación global del audio, que finalmente se utiliza para
clasificarlo.

# 7. Ejemplo 5 — Predicción de series temporales

Multi-Head Attention también puede utilizarse para analizar series
temporales.

Una serie temporal está formada por valores registrados a lo largo
del tiempo.

Cada instante puede representarse mediante un vector de
características y tratarse como un token.

Multi-Head Attention permite que cada instante pueda relacionarse
con otros momentos de la secuencia para obtener información
contextual y realizar predicciones.

### Representación de la serie temporal

Cada instante de tiempo contiene 4 características.

La entrada tendrá la forma:

[batch_size, sequence_length, feature_dim]

Por ejemplo:

[4, 50, 4]

representa 4 secuencias, cada una con 50 instantes temporales y
4 características por instante.

In [24]:
import torch
import torch.nn as nn

batch_size = 4
sequence_length = 50
feature_dim = 4
num_heads = 4

# Datos de la serie temporal
series = torch.randn(
    batch_size,
    sequence_length,
    feature_dim
)

print("Entrada:", series.shape)

Entrada: torch.Size([4, 50, 4])


### Embedding temporal

Antes de aplicar Multi-Head Attention podemos proyectar las
características de cada instante a un espacio de mayor dimensión.

Esto permite trabajar con una representación más rica de cada
instante temporal.

In [25]:
embedding_dim = 64

embedding = nn.Linear(
    feature_dim,
    embedding_dim
)

x = embedding(series)

print("Embeddings:", x.shape)

Embeddings: torch.Size([4, 50, 64])


### Multi-Head Self-Attention

Utilizamos Self-Attention porque Q, K y V provienen de la misma
secuencia temporal.

Por lo tanto:

Q = X
K = X
V = X

Cada instante puede relacionarse con los demás instantes de la
secuencia.

In [26]:
attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)

output, weights = attention(
    x,
    x,
    x
)

print("Entrada:", x.shape)
print("Salida:", output.shape)

Entrada: torch.Size([4, 50, 64])
Salida: torch.Size([4, 50, 64])


### Predicción

Después de aplicar Multi-Head Attention obtenemos una representación
contextual para cada instante.

En este ejemplo utilizaremos la representación del último instante
para realizar la predicción del siguiente valor.

In [27]:
last_output = output[:, -1, :]

print("Último instante:", last_output.shape)

Último instante: torch.Size([4, 64])


In [28]:
predictor = nn.Linear(
    embedding_dim,
    1
)

prediction = predictor(
    last_output
)

print("Predicción:", prediction.shape)

Predicción: torch.Size([4, 1])


### Código completo

El siguiente ejemplo muestra una implementación simplificada de
Multi-Head Attention para utilizar una secuencia temporal y predecir
su siguiente valor.

In [29]:
import torch
import torch.nn as nn


# Parámetros
batch_size = 4
sequence_length = 50
feature_dim = 4
embedding_dim = 64
num_heads = 4


# Datos de la serie temporal
series = torch.randn(
    batch_size,
    sequence_length,
    feature_dim
)


# Embedding
embedding = nn.Linear(
    feature_dim,
    embedding_dim
)

x = embedding(series)


# Multi-Head Attention
attention = nn.MultiheadAttention(
    embed_dim=embedding_dim,
    num_heads=num_heads,
    batch_first=True
)

output, weights = attention(
    x,
    x,
    x
)


# Representación del último instante
last_output = output[:, -1, :]


# Predictor
predictor = nn.Linear(
    embedding_dim,
    1
)

prediction = predictor(
    last_output
)


print("Entrada:", series.shape)
print("Embeddings:", x.shape)
print("Attention:", output.shape)
print("Último instante:", last_output.shape)
print("Predicción:", prediction.shape)

Entrada: torch.Size([4, 50, 4])
Embeddings: torch.Size([4, 50, 64])
Attention: torch.Size([4, 50, 64])
Último instante: torch.Size([4, 64])
Predicción: torch.Size([4, 1])


### ¿Cómo se utiliza Multi-Head Attention?

En este ejemplo, cada instante temporal se considera un token.

Multi-Head Attention permite que cada instante pueda relacionarse
con otros momentos de la secuencia y determinar qué información
histórica resulta relevante.

La representación contextual obtenida se utiliza posteriormente
para predecir el siguiente valor de la serie temporal.

### Comparación con RNN y LSTM

RNN y LSTM procesan normalmente la secuencia de forma recurrente,
manteniendo información del estado anterior.

Multi-Head Attention utiliza otro enfoque: permite relacionar
directamente diferentes posiciones de la secuencia mediante los
pesos de atención.

Esto facilita modelar relaciones entre elementos alejados dentro
de la secuencia.